# 批量结构分析工作流

**主要功能：**
- 批量分析多个蛋白质结构
- 结构质量评估和比较
- 结构相似性分析
- 生成综合分析报告

**输入：**
- 多个PDB结构文件
- 或包含PDB文件的目录

**输出：**
- 结构质量报告
- 结构相似性矩阵
- 聚类分析结果
- 可视化图表

**系统要求：**
- 已安装结构分析工具
- 约 5-10 GB 磁盘空间
- 适合大规模结构数据集

## 1. 环境设置与依赖安装

In [ ]:
# 使用共享工具初始化环境（自动安装依赖、设置路径）
import sys
from pathlib import Path

# 添加protflow到路径
project_root = Path.cwd()
while not (project_root / 'src' / 'protflow').exists() and project_root != project_root.parent:
    project_root = project_root.parent

if (project_root / 'src').exists():
    src_dir = str(project_root / 'src')
    if src_dir not in sys.path:
        sys.path.insert(0, src_dir)
    print(f"✓ protflow 路径: {src_dir}")

# 导入并设置环境
from protflow.utils.notebook_utils import setup_analysis_notebook

# 设置环境（自动检查和安装依赖）
paths = setup_analysis_notebook(work_dir_name='batch_analysis_runs')

PROJECT_ROOT = paths['PROJECT_ROOT']
WORK_DIR = paths['WORK_DIR']
DATA_DIR = paths['DATA_DIR']

# 导入常用库
import pandas as pd
import matplotlib.pyplot as plt
from Bio import PDB
from tqdm import tqdm
import numpy as np

print(f"\n✓ 环境初始化完成")
print(f"  工作目录: {WORK_DIR}")

## 2. 输入结构文件收集

In [ ]:
# 使用后端模块收集结构文件（所有业务逻辑在后端）
from protflow.core.structure_analysis import collect_structure_files
from pathlib import Path

# 输入路径
input_structures = "path/to/pdb/files"  # 替换为您的路径

structure_info = collect_structure_files(Path(input_structures))


## 3. 结构质量评估

In [ ]:
# 使用后端模块进行批量质量评估（所有业务逻辑在后端）
from protflow.core.structure_analysis import batch_quality_assessment
import pandas as pd

# 运行批量质量评估
if structure_info:
    quality_results = batch_quality_assessment(structure_info)
    
    # 保存结果
    results_df = pd.DataFrame(quality_results)
    quality_file = WORK_DIR / 'structure_quality.csv'
    results_df.to_csv(quality_file, index=False)
    print(f"\n✓ 质量评估结果保存: {quality_file}")
else:
    quality_results = []
    print("⚠️ 没有结构文件可供评估")


## 4. 结构相似性分析

In [ ]:
# 使用后端模块进行相似性分析（所有业务逻辑在后端）
from protflow.core.structure_analysis import batch_similarity_analysis
import numpy as np

# 运行相似性分析
if quality_results and len(quality_results) >= 2:
    similarity_data = batch_similarity_analysis(quality_results)
    
    # 保存结果
    if similarity_data and 'similarity_results_df' in similarity_data:
        similarity_file = WORK_DIR / 'structure_similarity.csv'
        similarity_data['similarity_results_df'].to_csv(similarity_file, index=False)
        print(f"\n✓ 相似性结果保存: {similarity_file}")
    
    if similarity_data and 'similarity_matrix_df' in similarity_data:
        matrix_file = WORK_DIR / 'similarity_matrix.csv'
        similarity_data['similarity_matrix_df'].to_csv(matrix_file)
        print(f"✓ 相似性矩阵保存: {matrix_file}")
else:
    similarity_data = None
    print("⚠️ 无法进行相似性分析: 需要至少2个有效结构")


## 5. 结构聚类分析

In [ ]:
# 使用后端模块进行聚类分析（所有业务逻辑在后端）
from protflow.core.structure_analysis import perform_clustering_analysis

# 执行聚类分析
if similarity_data:
    clustering_results = perform_clustering_analysis(similarity_data)
    
    # 保存结果
    if clustering_results:
        clustering_df = pd.DataFrame({
            'structure_name': similarity_data['structure_names'],
            'cluster': clustering_results['labels'],
            'pca1': clustering_results['pca_result'][:, 0],
            'pca2': clustering_results['pca_result'][:, 1]
        })
        clustering_file = WORK_DIR / 'structure_clustering.csv'
        clustering_df.to_csv(clustering_file, index=False)
        print(f"\n✓ 聚类结果保存: {clustering_file}")
else:
    clustering_results = None
    print("⚠️ 无法进行聚类分析: 缺少相似性数据")


## 6. 结果可视化

In [ ]:
def visualize_analysis_results(quality_results, similarity_data, clustering_results):
    """可视化分析结果"""
    
    try:
        import matplotlib.pyplot as plt
        import seaborn as sns
        
        # 设置图形样式
        plt.style.use('default')
        fig = plt.figure(figsize=(20, 16))
        
        # 1. 结构质量分布
        ax1 = plt.subplot(2, 3, 1)
        successful_results = [r for r in quality_results if r['status'] == 'success']
        
        if successful_results:
            residues = [r['num_residues'] for r in successful_results]
            ax1.hist(residues, bins=20, alpha=0.7, color='skyblue', edgecolor='black')
            ax1.set_xlabel('残基数量')
            ax1.set_ylabel('结构数量')
            ax1.set_title('结构大小分布')
            ax1.axvline(np.mean(residues), color='red', linestyle='--', 
                       label=f'平均值: {np.mean(residues):.0f}')
            ax1.legend()
        
        # 2. 质量状态统计
        ax2 = plt.subplot(2, 3, 2)
        status_counts = {}
        for result in quality_results:
            status = result['status']
            status_counts[status] = status_counts.get(status, 0) + 1
        
        if status_counts:
            statuses = list(status_counts.keys())
            counts = list(status_counts.values())
            colors = ['green', 'orange', 'red', 'gray']
            
            bars = ax2.bar(statuses, counts, color=colors[:len(statuses)], alpha=0.7)
            ax2.set_xlabel('状态')
            ax2.set_ylabel('数量')
            ax2.set_title('结构质量状态分布')
            
            # 添加数值标签
            for bar, count in zip(bars, counts):
                ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                        str(count), ha='center', va='bottom')
        
        # 3. 相似性矩阵热图
        if similarity_data:
            ax3 = plt.subplot(2, 3, 3)
            similarity_matrix = similarity_data['similarity_matrix']
            
            sns.heatmap(similarity_matrix, 
                       annot=False, 
                       cmap='viridis', 
                       square=True,
                       ax=ax3,
                       cbar_kws={'label': '相似性分数'})
            
            ax3.set_title('结构相似性矩阵')
            
            # 简化标签
            if len(similarity_matrix) <= 20:  # 只在结构数较少时显示标签
                labels = [name[:10] + '...' if len(name) > 10 else name 
                         for name in similarity_data['structure_names']]
                ax3.set_xticklabels(labels, rotation=45, ha='right')
                ax3.set_yticklabels(labels, rotation=0)
        
        # 4. PCA散点图
        if clustering_results:
            ax4 = plt.subplot(2, 3, 4)
            pca_result = clustering_results['pca_result']
            labels = clustering_results['labels']
            
            # 为不同聚类使用不同颜色
            unique_labels = np.unique(labels)
            colors = plt.cm.Set3(np.linspace(0, 1, len(unique_labels)))
            
            for i, (label, color) in enumerate(zip(unique_labels, colors)):
                mask = labels == label
                ax4.scatter(pca_result[mask, 0], pca_result[mask, 1],
                           c=[color], label=f'聚类 {label + 1}',
                           s=50, alpha=0.7)
            
            ax4.set_xlabel('PC1')
            ax4.set_ylabel('PC2')
            ax4.set_title('结构PCA可视化')
            ax4.legend()
            ax4.grid(True, alpha=0.3)
        
        # 5. RMSD分布
        if similarity_data:
            ax5 = plt.subplot(2, 3, 5)
            rmsd_values = []
            
            for i in range(len(similarity_data['structure_names'])):
                for j in range(i + 1, len(similarity_data['structure_names'])):
                    rmsd = similarity_data['rmsd_matrix'].iloc[i, j]
                    if not pd.isna(rmsd):
                        rmsd_values.append(rmsd)
            
            if rmsd_values:
                ax5.hist(rmsd_values, bins=20, alpha=0.7, color='lightcoral', edgecolor='black')
                ax5.set_xlabel('RMSD (Å)')
                ax5.set_ylabel('结构对数量')
                ax5.set_title('RMSD分布')
                ax5.axvline(np.mean(rmsd_values), color='darkred', linestyle='--',
                           label=f'平均值: {np.mean(rmsd_values):.1f} Å')
                ax5.legend()
        
        # 6. 聚类大小分布
        if clustering_results:
            ax6 = plt.subplot(2, 3, 6)
            cluster_info = clustering_results['cluster_info']
            
            cluster_sizes = [len(members) for members in cluster_info.values()]
            
            if cluster_sizes:
                bars = ax6.bar(range(len(cluster_sizes)), sorted(cluster_sizes, reverse=True),
                              color='lightgreen', alpha=0.7)
                ax6.set_xlabel('聚类编号')
                ax6.set_ylabel('结构数量')
                ax6.set_title('聚类大小分布')
                
                # 添加数值标签
                for bar, size in zip(bars, sorted(cluster_sizes, reverse=True)):
                    ax6.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                            str(size), ha='center', va='bottom')
        
        plt.tight_layout()
        
        # 保存图形
        plot_file = WORK_DIR / 'batch_analysis_results.png'
        plt.savefig(plot_file, dpi=300, bbox_inches='tight')
        print(f"✓ 分析可视化结果保存: {plot_file}")
        
        plt.show()
        
    except ImportError:
        print("⚠️ 需要安装 matplotlib 和 seaborn 进行可视化")
        print("运行: pip install matplotlib seaborn")
    
    except Exception as e:
        print(f"❌ 可视化失败: {e}")

# 运行可视化
if quality_results:
    visualize_analysis_results(quality_results, similarity_data, clustering_results)
else:
    print("⚠️ 没有分析结果可供可视化")

## 7. 综合分析报告

In [ ]:
def generate_comprehensive_report(quality_results, similarity_data, clustering_results):
    """生成综合分析报告"""
    
    report_file = WORK_DIR / 'analysis_report.txt'
    
    with open(report_file, 'w', encoding='utf-8') as f:
        f.write("蛋白质结构批量分析报告\n")
        f.write("=" * 50 + "\n\n")
        
        # 基本信息
        f.write(f"分析时间: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"工作目录: {WORK_DIR}\n\n")
        
        # 质量评估结果
        f.write("1. 结构质量评估\n")
        f.write("-" * 30 + "\n")
        
        status_counts = {}
        for result in quality_results:
            status = result['status']
            status_counts[status] = status_counts.get(status, 0) + 1
        
        f.write(f"总结构数: {len(quality_results)}\n")
        for status, count in status_counts.items():
            percentage = (count / len(quality_results)) * 100
            f.write(f"  {status}: {count} ({percentage:.1f}%)\n")
        
        # 成功结构的统计
        successful = [r for r in quality_results if r['status'] == 'success']
        if successful:
            residues = [r['num_residues'] for r in successful]
            atoms = [r['num_atoms'] for r in successful]
            
            f.write(f"\n成功结构统计:")
            f.write(f"  平均残基数: {np.mean(residues):.1f}\n")
            f.write(f"  残基数范围: {min(residues)} - {max(residues)}\n")
            f.write(f"  平均原子数: {np.mean(atoms):.1f}\n")
            f.write(f"  原子数范围: {min(atoms)} - {max(atoms)}\n")
        
        # 相似性分析结果
        if similarity_data:
            f.write(f"\n2. 结构相似性分析\n")
            f.write("-" * 30 + "\n")
            
            n_structures = len(similarity_data['structure_names'])
            f.write(f"分析结构数: {n_structures}\n")
            f.write(f"结构对总数: {n_structures * (n_structures - 1) // 2}\n")
            
            # 相似性统计
            similarities = []
            rmsds = []
            for i in range(n_structures):
                for j in range(i + 1, n_structures):
                    sim = similarity_data['similarity_matrix'].iloc[i, j]
                    rmsd = similarity_data['rmsd_matrix'].iloc[i, j]
                    similarities.append(sim)
                    if not pd.isna(rmsd):
                        rmsds.append(rmsd)
            
            if similarities:
                f.write(f"平均相似性: {np.mean(similarities):.3f}\n")
                f.write(f"相似性范围: {np.min(similarities):.3f} - {np.max(similarities):.3f}\n")
            
            if rmsds:
                f.write(f"平均RMSD: {np.mean(rmsds):.2f} Å\n")
                f.write(f"RMSD范围: {np.min(rmsds):.2f} - {np.max(rmsds):.2f} Å\n")
        
        # 聚类结果
        if clustering_results:
            f.write(f"\n3. 结构聚类分析\n")
            f.write("-" * 30 + "\n")
            
            f.write(f"聚类数: {clustering_results['n_clusters']}\n")
            f.write(f"聚类质量评分: {clustering_results['score']:.3f}\n\n")
            
            cluster_info = clustering_results['cluster_info']
            for cluster_id, members in cluster_info.items():
                f.write(f"聚类 {cluster_id + 1}: {len(members)} 个结构\n")
                for member in members:
                    f.write(f"  - {member}\n")
                f.write("\n")
        
        # 结果文件位置
        f.write(f"\n4. 结果文件\n")
        f.write("-" * 30 + "\n")
        f.write(f"质量评估结果: {WORK_DIR / 'structure_quality.csv'}\n")
        
        if similarity_data:
            f.write(f"相似性结果: {WORK_DIR / 'structure_similarity.csv'}\n")
            f.write(f"相似性矩阵: {WORK_DIR / 'similarity_matrix.csv'}\n")
        
        if clustering_results:
            f.write(f"聚类结果: {WORK_DIR / 'structure_clustering.csv'}\n")
        
        f.write(f"可视化图表: {WORK_DIR / 'batch_analysis_results.png'}\n")
        
        # 建议和下一步
        f.write(f"\n5. 建议\n")
        f.write("-" * 30 + "\n")
        
        if clustering_results:
            f.write("基于聚类结果，建议对每个聚类进行代表性结构分析:")
            cluster_info = clustering_results['cluster_info']
            for cluster_id, members in cluster_info.items():
                if len(members) > 0:
                    f.write(f"  - 聚类 {cluster_id + 1}: 推荐分析 {members[0]}\n")
        
        if successful:
            # 找出最大和最小的结构
            largest = max(successful, key=lambda x: x['num_residues'])
            smallest = min(successful, key=lambda x: x['num_residues'])
            
            f.write(f"\n特殊结构推荐:")
            f.write(f"  - 最大结构: {largest['protein_name']} ({largest['num_residues']} 残基)\n")
            f.write(f"  - 最小结构: {smallest['protein_name']} ({smallest['num_residues']} 残基)\n")
        
        f.write(f"\n可使用以下工具进行进一步分析:")
        f.write(f"  - 01_protein_structure_prediction.ipynb: 结构预测\n")
        f.write(f"  - 02_pocket_detection_p2rank.ipynb: 口袋检测\n")
        f.write(f"  - 12_structure_alignment_dali.ipynb: 结构比对\n")
    
    print(f"✓ 综合分析报告生成: {report_file}")
    
    # 显示报告摘要
    print("\n=== 分析报告摘要 ===")
    print(f"总结构数: {len(quality_results)}")
    
    successful_count = len([r for r in quality_results if r['status'] == 'success'])
    print(f"成功结构: {successful_count}")
    
    if similarity_data:
        print(f"相似性分析: {len(similarity_data['structure_names'])} 个结构")
    
    if clustering_results:
        print(f"聚类分析: {clustering_results['n_clusters']} 个聚类")
    
    print(f"详细报告见: {report_file}")

# 生成报告
if quality_results:
    generate_comprehensive_report(quality_results, similarity_data, clustering_results)
else:
    print("⚠️ 没有分析结果，无法生成报告")

## 8. 下一步操作

完成批量结构分析后，您可以：

1. **详细分析**:
   - 选择代表性结构进行口袋检测 (`02_pocket_detection_p2rank.ipynb`)
   - 对关键结构进行分子对接 (`03_ligand_docking_vina.ipynb`)
   - 使用DALI进行更精确的结构比对 (`12_structure_alignment_dali.ipynb`)

2. **聚类分析**:
   - 对每个聚类的代表结构进行功能分析
   - 研究聚类间的功能差异
   - 分析聚类与蛋白质功能的关系

3. **质量控制**:
   - 基于质量评估结果过滤低质量结构
   - 重新预测或优化质量较差的结构
   - 建立结构质量标准

4. **功能注释**:
   - 结合序列分析工具进行功能预测
   - 比较结构相似性与序列相似性的关系
   - 研究结构保守性与功能的关系

**结果文件位置:**
- 质量评估: `{WORK_DIR}/structure_quality.csv`
- 相似性矩阵: `{WORK_DIR}/similarity_matrix.csv`
- 聚类结果: `{WORK_DIR}/structure_clustering.csv`
- 可视化图表: `{WORK_DIR}/batch_analysis_results.png`
- 综合报告: `{WORK_DIR}/analysis_report.txt`

**注意:**
- 大规模结构分析需要较长的计算时间
- 建议先在小规模数据集上测试参数
- 结构相似性分析对内存要求较高
- 结果解释需要结合生物学背景知识